In [0]:
import pandas as pd                                                     # Import pandas for data cleaning
import seaborn as sns                                                   # Import Seaborn for visualization
import matplotlib.pyplot as plt                                         # Import matplotlib library for visualization
import plotly.express as px                                             # Import plotly library for visualization
import plotly.graph_objects as go

from pyspark.sql import functions as F
from pyspark.sql.functions import col                                   # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev,count          # MathsFunctions
from pyspark.sql.functions import to_date,month,year,datediff           # DateFunctions
from pyspark.sql.functions import abs                                   # OtherFunctions

In [0]:
orders = spark.read.table("samplesuperstore.bronzedata.orders")
people = spark.read.table("samplesuperstore.bronzedata.people")
returns = spark.read.table("samplesuperstore.bronzedata.returns")

orders_customerdetails = spark.read.table("samplesuperstore.silverdata.orders_customerdetails")
orders_geordetails = spark.read.table("samplesuperstore.silverdata.orders_geordetails")
orders_orderdetails = spark.read.table("samplesuperstore.silverdata.orders_orderdetails")
orders_prime = spark.read.table("samplesuperstore.silverdata.orders_prime")
orders_productdetails = spark.read.table("samplesuperstore.silverdata.orders_productdetails")

In [0]:
# FORMATING THE COLUMNS NAMES  
    # For Order table : replace spaces with underscores, lowercase
clean_cols = [c.replace(" ", "_").lower() for c in orders.columns]
for old, new in zip(orders.columns, clean_cols):
    orders = orders.withColumnRenamed(old, new)

    # For people table : replace spaces with underscores, lowercase
clean_cols = [c.replace(" ", "_").lower() for c in people.columns]
for old, new in zip(people.columns, clean_cols):
    people = people.withColumnRenamed(old, new)

    # For returns table : replace spaces with underscores, lowercase
clean_cols = [c.replace(" ", "_").lower() for c in returns.columns]
for old, new in zip(returns.columns, clean_cols):
    returns = returns.withColumnRenamed(old, new)

# display(summary_df)

In [0]:
# Cell 3: Summary statistics for Orders table
from pyspark.sql.functions import col, mean, min, max, stddev, count, when, expr

# Example: Sales, Profit, Quantity columns
summary_orders = orders.select(
    mean(col("Sales")).alias("Sales_Mean"),
    expr("percentile_approx(Sales, 0.5)").alias("Sales_Median"),
    min(col("Sales")).alias("Sales_Min"),
    max(col("Sales")).alias("Sales_Max"),
    stddev(col("Sales")).alias("Sales_StdDev"),
    count(when(col("Sales").isNull(), 1)).alias("Sales_Nulls"),

    mean(col("Profit")).alias("Profit_Mean"),
    expr("percentile_approx(Profit, 0.5)").alias("Profit_Median"),
    min(col("Profit")).alias("Profit_Min"),
    max(col("Profit")).alias("Profit_Max"),
    stddev(col("Profit")).alias("Profit_StdDev"),
    count(when(col("Profit").isNull(), 1)).alias("Profit_Nulls"),

    mean(col("Quantity")).alias("Quantity_Mean"),
    expr("percentile_approx(Quantity, 0.5)").alias("Quantity_Median"),
    min(col("Quantity")).alias("Quantity_Min"),
    max(col("Quantity")).alias("Quantity_Max"),
    stddev(col("Quantity")).alias("Quantity_StdDev"),
    count(when(col("Quantity").isNull(), 1)).alias("Quantity_Nulls")
)
display(summary_orders)

Sales_Mean,Sales_Median,Sales_Min,Sales_Max,Sales_StdDev,Sales_Nulls,Profit_Mean,Profit_Median,Profit_Min,Profit_Max,Profit_StdDev,Profit_Nulls,Quantity_Mean,Quantity_Median,Quantity_Min,Quantity_Max,Quantity_StdDev,Quantity_Nulls
229.85800083049827,54.384,0.444,22638.48,623.2451005086805,0,28.656896307784663,8.6436,-6599.978,8399.976,234.2601076909573,0,3.789573744246548,3.0,1.0,14.0,2.2251096911413994,0


In [0]:
#HANDLING NULLS
def nulls_summary(table_name):
    df = spark.table(table_name)
    total_records = df.count()
    summary = []
    for col in df.columns:
        nulls = df.filter(F.col(col).isNull()).count()
        not_nulls = total_records - nulls
        summary.append((table_name, col, total_records, not_nulls, nulls))
    return summary

    # Collect summaries for all three tables
orders_summary  = nulls_summary("samplesuperstore.bronzedata.orders")
people_summary  = nulls_summary("samplesuperstore.bronzedata.people")
returns_summary = nulls_summary("samplesuperstore.bronzedata.returns")

    # Combine into one DataFrame
summary_df = spark.createDataFrame(
    orders_summary + people_summary + returns_summary,
    ["Table", "Column", "TotalRecords", "NotNulls", "Nulls"]
)
summary_df.write.mode("overwrite").saveAsTable("samplesuperstore.silverdata.eda_handlingnulls")

In [0]:
#HANDLING OUTLINERS 
stats = orders.select(
    F.mean("sales").alias("mean_sales"),
    F.stddev("sales").alias("std_sales"),
    F.mean("profit").alias("mean_profit"),
    F.stddev("profit").alias("std_profit")
).collect()[0]

mean_sales, std_sales = stats["mean_sales"], stats["std_sales"]
mean_profit, std_profit = stats["mean_profit"], stats["std_profit"]

orders_outliers = orders.withColumn(
    "SalesOutlier",
    F.when((F.col("sales") > mean_sales + 3*std_sales) | (F.col("sales") < mean_sales - 3*std_sales), 1).otherwise(0)
).withColumn(
    "ProfitOutlier",
    F.when((F.col("profit") > mean_profit + 3*std_profit) | (F.col("profit") < mean_profit - 3*std_profit), 1).otherwise(0)
)
orders_outliers.write.mode("overwrite").saveAsTable("samplesuperstore.silverdata.orders_outliers")

In [0]:
from pyspark.sql import functions as F

def fill_nulls_with_mode_mean(df):
    for col, dtype in df.dtypes:
        if dtype == "string":
            mode_val = df.groupBy(col).count().orderBy(F.desc("count")).first()[0]
            df = df.fillna({col: mode_val})
        elif dtype in ["double","int","long","float","decimal"]:
            mean_val = df.select(F.mean(F.col(col))).first()[0]
            if mean_val is not None:
                df = df.fillna({col: mean_val})
    return df

# Apply cleaning on Bronze tables
df_orders  = fill_nulls_with_mode_mean(orders)
df_people  = fill_nulls_with_mode_mean(people)
df_returns = fill_nulls_with_mode_mean(returns)